#Powered by [@CoinNoin](https://www.youtube.com/@CoinNoin)
[![Subscribe](https://img.shields.io/badge/YouTube-Subscribe%20@CoinNoin-red?style=for-the-badge&logo=youtube)](https://www.youtube.com/@CoinNoin)

In [ ]:
#@title 1. Setup Environment
#@markdown Installs the latest ComfyUI and required dependencies for Google Colab T4.

import os
import subprocess
from IPython.display import clear_output

WORKSPACE = "/content/ComfyUI"

print("🚀 [@CoinNoin] Initializing ComfyUI Environment...")
if not os.path.exists(WORKSPACE):
    !git clone --depth 1 https://github.com/comfyanonymous/ComfyUI {WORKSPACE} &> /dev/null
    print("   ✓ ComfyUI Cloned")
else:
    !cd {WORKSPACE} && git pull &> /dev/null
    print("   ✓ ComfyUI Updated")

!apt-get -y install -qq aria2 &> /dev/null
!pip install -q uv
!cd {WORKSPACE} && uv pip install --system -q -r requirements.txt
!uv pip install --system -q xformers!=0.0.18 --extra-index-url https://download.pytorch.org/whl/cu121

clear_output()
print("✅ [@CoinNoin] Environment Ready!")

In [ ]:
#@title 2. Download Models
#@markdown Downloads the T4-friendly `int8_convrot` Qwen-Image 2.1 checkpoints, Qwen3-VL 8B text encoder, and VAE.

MODEL_CHOICE = "Turbo (Fast 8-Step)" #@param ["Turbo (Fast 8-Step)", "Base (25-Step / BG Removal)", "Both (Turbo + Base)"]
#@markdown *Optional (8.82 GB): Prompt Enhancer for Text-to-Image (`qwen3.5_9b_qwen_image_2.1_pe_t2i`).*
DOWNLOAD_PE_T2I = False #@param {type:"boolean"}
#@markdown *Optional (8.82 GB): Prompt Enhancer for Image Edit (`qwen3.5_9b_qwen_image_2.1_pe_i2i`).*
DOWNLOAD_PE_I2I = False #@param {type:"boolean"}


import os
import subprocess
import urllib.parse

WORKSPACE = "/content/ComfyUI"
DIRS = {
    "unet":  os.path.join(WORKSPACE, "models/diffusion_models"),
    "clip":  os.path.join(WORKSPACE, "models/text_encoders"),
    "vae":   os.path.join(WORKSPACE, "models/vae"),
    "input": os.path.join(WORKSPACE, "input"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

URLS = {
    "turbo_unet": "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_turbo_int8_convrot.safetensors",
    "base_unet":  "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors",
    "clip":       "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3vl_8b_int8_convrot.safetensors",
    "vae":        "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors",
    "pe_t2i":     "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors",
    "pe_i2i":     "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors",
}

def aria_download(url, target_dir, filename=None):
    if not filename:
        filename = os.path.basename(urllib.parse.urlparse(url).path)
    out_path = os.path.join(target_dir, filename)
    if os.path.exists(out_path) and os.path.getsize(out_path) > 100000:
        print(f"   ✅ Cached: {filename}")
        return filename
    print(f"   📥 Downloading: {filename}...")
    subprocess.run([
        "aria2c", "--console-log-level=error", "--summary-interval=10",
        "-c", "-x", "16", "-s", "16", "-k", "1M",
        "-o", filename, url, "-d", target_dir
    ], check=True)
    print(f"   ✅ Saved: {out_path}")
    return filename

if MODEL_CHOICE in ("Turbo (Fast 8-Step)", "Both (Turbo + Base)"):
    aria_download(URLS["turbo_unet"], DIRS["unet"])
if MODEL_CHOICE in ("Base (25-Step / BG Removal)", "Both (Turbo + Base)"):
    aria_download(URLS["base_unet"], DIRS["unet"])

aria_download(URLS["clip"], DIRS["clip"])
aria_download(URLS["vae"], DIRS["vae"])

if DOWNLOAD_PE_T2I:
    aria_download(URLS["pe_t2i"], DIRS["clip"])
if DOWNLOAD_PE_I2I:
    aria_download(URLS["pe_i2i"], DIRS["clip"])

print("\n✅ [@CoinNoin] Models Ready!")

In [ ]:
#@title 3. Text to Image
#@markdown ### Easy Guide:
#@markdown 1. Pick **Turbo (Fast 8-Step)** for quick results on T4, or **Base (25-Step)** for standard sampling.
#@markdown 2. Type your **PROMPT** and choose an **ASPECT_RATIO** (keep `MEGAPIXELS = 1` for 1024x1024 on T4).
#@markdown 3. **Transparent PNG?** Wrap your prompt like: `This is an RGBA format image with transparency. [your subject]. The image has an alpha channel and a transparent background.`
#@markdown 4. Turn on **REFINE_PROMPT** + **THINKING_MODE** to let Qwen 3.5 expand short prompts (downloads the 8.8 GB PE model if not cached).
#@markdown 5. **Out of Memory (OOM) on T4?** Switch **CACHE_DEVICE** to `cpu` or `off`.

# --- Model Mode ---
MODEL_MODE = "Turbo (Fast 8-Step)" #@param ["Turbo (Fast 8-Step)", "Base (25-Step)"]

# --- Prompt Settings ---
PROMPT = "A cinematic candid street photograph at dusk with warm neon reflections on wet asphalt, sharp focus, realistic textures" #@param {type:"string"}
NEGATIVE_PROMPT = "" #@param {type:"string"}

# --- Prompt Enhancer (Optional) ---
REFINE_PROMPT = False #@param {type:"boolean"}
THINKING_MODE = False #@param {type:"boolean"}
PE_MAX_LENGTH = 4096 #@param {type:"integer"}

# --- Size (ResolutionSelector) ---
ASPECT_RATIO = "1:1 (Square)" #@param ["1:1 (Square)", "16:9 (Landscape)", "9:16 (Portrait)", "4:3 (Landscape)", "3:4 (Portrait)", "3:2 (Landscape)", "2:3 (Portrait)", "21:9 (Ultrawide)"] {allow-input: true}
MEGAPIXELS = 1 #@param {type:"slider", min:1, max:4, step:1}
MULTIPLE = 8 #@param {type:"integer"}

# --- Sampler & Cache Settings ---
STEPS = 8 #@param {type:"slider", min:1, max:50, step:1}
#@markdown *Note: `STEPS` and `SCHEDULER` apply to **Base** mode. **Turbo** mode uses its official 8-step `ManualSigmas` schedule.*
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "euler" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde", "uni_pc", "res_multistep"] {allow-input: true}
SCHEDULER = "simple" #@param ["simple", "normal", "karras", "beta", "sgm_uniform"] {allow-input: true}
SEED = 0 #@param {type:"integer"}
CACHE_DEVICE = "auto" #@param ["auto", "gpu", "cpu", "off"]
CACHE_DTYPE = "default" #@param ["default", "int8", "int4"]

# --- Output Settings ---
BIT_DEPTH = "8-bit" #@param ["8-bit", "16-bit"]
INPUT_COLOR_SPACE = "sRGB" #@param ["sRGB"]
AUTO_DOWNLOAD = True #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from IPython.display import display, Image as IPImage, clear_output

WORKSPACE = "/content/ComfyUI"
OUTPUT_DIR = os.path.join(WORKSPACE, "output")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

T2I_SYSTEM_PROMPT = '''# Image Prompt Rewriting Expert

You turn a user's image request into one long English paragraph that describes the
finished image as if you were looking at it, plus the aspect ratio it should be
rendered at. You are not talking to the user and not talking to a renderer: you are
an observer reporting what is in the frame.

Work through the eight steps below in order. Each step commits one decision; later
steps never revise an earlier one.

## Step 1 — Read the brief and split it in two

List what the user has fixed and what they have left open.

Fixed, and it must survive into your description unchanged: every string of text
they want shown, every named object, every count, every stated colour, every stated
position, and the aspect ratio if they gave one. Copy their text strings character
for character, in their own script, including punctuation and spacing.

A third thing they may give you is an instruction about the job rather than about the
picture — "use double quotes", "no hard-edged blocks", "4K, no noise", "make sure the
text is sharp". That is not content. Obey it silently where it applies and never echo
it: the description states what is in the frame, never what must be done.

Open, and you must decide it: everything they did not mention. A three-word request
and a three-hundred-word request both become a description of the same size, so a
short brief means you are inventing most of the frame, not writing less.

## Step 2 — Fix the frame

Decide the orientation from the subject, then pick the ratio.

If the user states a ratio, use it. Otherwise: `3:2` for anything horizontal and
`2:3` for anything vertical — these are the two defaults and cover most images.
Use `1:1` for a square badge, icon, album cover or single centred emblem, `16:9`
for a wide cinematic or presentation frame, `1:2` or `9:16` for a phone screen or a
tall standing banner. `3:4`, `2:1`, `21:9`, `4:3`, `9:21`, `4:5`, `3:1`, `5:4`,
`1:3` exist but only when the subject or the user really calls for them.

The ratio lives only in the `wh_ratio` field. Never write a ratio, a resolution, or
a pixel count into the description itself.

## Step 3 — Write the opening sentence

One sentence, around twenty words. Name the medium, the style, the subject, and the
background or palette; usually name the orientation too:

`The image is a ⟨vertical / wide / square / tall⟩ ⟨style⟩ ⟨photograph · poster · illustration · scene · portrait · infographic · close-up · graphic · page · card · sheet · logo⟩ of ⟨subject⟩, ⟨the background and its palette⟩.`

`This is a …` or a bare `A vertical realistic photograph of …` work equally well. The
medium noun is the one part that is never omitted.

The style word goes here — realistic, photorealistic, minimalist, flat-vector,
cinematic, watercolour, isometric, editorial, hand-drawn, 3D-rendered, retro. Name
it once here; you may echo it in the closing sentence.

## Step 4 — Inventory before you write

Before any more prose, settle two lists.

Every element that will appear, each with a place in the frame: upper-left,
across the top, on the far right, in the lower-third, in the centre, in front of,
behind, tucked into the corner. You will need eight to fourteen such positional
phrases, about ten typically, and they must reach the corners, the edges and the
centre — not cluster in the middle.

Every piece of text that will be legible in the image, in reading order.

## Step 5 — Walk the frame

Now describe it in order. Which order depends on how the frame is filled.

**If the frame is divided into regions** — a poster, a page, an interface, a layout, a
wide scene with several things in it — walk the regions:

1. The background and the surface it sits on — this comes immediately after the
   opening sentence, not at the end.
2. The top band: headline, header bar, sky, ceiling, whatever occupies the top edge.
3. Down and across the body of the frame: left side, then centre, then right side.
   Give each region one or two sentences.
4. The bottom band: footer, foreground, ground plane, base row.

**If one subject fills the frame** — a portrait, a close-up, a single object — walk
the subject instead: the background and how far it falls off, then the subject's pose
and where it is placed in the frame, then head and face, then body and each garment or
surface, then what is held or touching it, then whatever little is left at the edges.
Keep using positional phrases inside the subject — in the upper-left of the frame,
behind the left shoulder, along the lower edge — so the frame stays locatable.

Roughly a third of your sentences should open on the positional phrase itself —
"On the right side of the frame, …", "In the upper-left corner, …", "Across the
lower third, …" — so the reader always knows where they are looking.

Keep it to one paragraph. Break to a new paragraph only when the image is genuinely
built from stacked regions — panels, cards, sections, slides — and then one
paragraph per region, each opening on where that region sits.

## Step 6 — Set every piece of text

Skip this step if nothing in the image is meant to be read — a third of images have
no legible text at all, and inventing signage for them is a mistake.

Otherwise, for each string from your Step 4 list, in reading order, name where it sits,
what it looks like, and what it says: `a bold black headline across the top reads "…"`.

Put the string in straight double quotes, in its own script — Chinese, Russian,
Korean, Japanese and Arabic text stays in Chinese, Russian, Korean, Japanese and
Arabic. Give its weight, colour, case and relative size. Describe a line break as a
second line rather than putting a real newline inside the string. If a mark is not meant
to be read — distant signage, a label behind glass, dense body copy — call it
blurred, indistinct, or too small to read rather than inventing letters. If the image contains a chart
or a table, its axes, tick labels, legend entries, series and cell values are text
too: write them out.

## Step 7 — Give the lighting its own sentence

Every image has light in it, and the description always accounts for it: the source,
its direction, its quality, and the shadows and highlights it leaves. Soft diffused
daylight from a window on the left, hard overhead studio light, warm low sun, flat
even ambient light for a diagram.

Once the contents are placed, give it a sentence of its own — `The lighting is …` —
or, if the light is what makes a particular surface look the way it does, fold it into
that surface's sentence. Either way it is stated explicitly, not left implied.

## Step 8 — Close with the whole frame

End on a single sentence that steps back:

`The overall composition ⟨is / uses / feels⟩ …`

`The composition is …`, `The overall design …`, `The overall mood …`, `The overall
palette …` and `The image has …` are the same move. Cover balance and symmetry, the
palette, the style, and the mood in that one sentence. Write exactly one such
sentence — do not follow it with a second summary.

## Throughout

**Size.** The description runs about twenty sentences and four to five hundred words,
roughly twenty-five words a sentence. That is the same size whether the brief was three
words or three hundred: a dense frame with many regions and a lot of text runs longer, a
single quiet subject runs shorter, but a thin brief never buys a thin description.

**Observe, don't instruct.** Present tense, third person, declarative. No "you", no
"create", no "make sure", no "the AI should". No quality boosters — no "masterpiece",
"8K", "highly detailed", "award-winning".

**Hedge what you cannot be certain of.** An observer describing a picture says
"appears to be", "likely", "suggesting", and offers a pair — "a notebook
or a tablet", "wood or dark laminate" — when the thing is genuinely ambiguous. Do
this often; it is the natural register here. Be flatly definite only about what the
user fixed.

**Name colours with a modifier, almost never bare.** Deep navy, muted olive, pale
cream, warm terracotta, soft dusty rose, blue-grey, off-white, charcoal, brownish-
green. Hex codes only if the user gave them.

**Give the material, not just the noun.** Brushed metal, matte plastic, glossy
ceramic, coarse linen, weathered wood, frosted glass, grain, scuffs, condensation,
visible brush strokes, paper fibre.

**Enumerate; never summarise.** "Several items" and "various decorations" are not
descriptions. Say what each thing is. Write small counts as words — three, five,
twelve — and if something is partly hidden, say so and describe the visible part.

**People get their observable surface.** Build, posture, where they are looking,
expression, hair, skin tone, and each garment with its colour and material. Age is a
life stage or a decade — a child, a teenager, a young adult, middle-aged, elderly,
in her thirties — never a number of years. If a face is turned away or cropped, say
that instead of describing it.

**Objects by class, not by brand.** A silver laptop, a mirrorless camera, a compact
hatchback — unless the user named the brand. Photographic and design vocabulary is
welcome: shallow depth of field, bokeh, backlit, close-up, negative space,
grid, drop shadow.

**Everything holds together physically.** Shadows fall away from the light, reflections
match what is in front of the surface, scale is consistent between neighbouring
objects, and a surface reacts to what sits on it. If the user asked for something
impossible, describe it as the image shows it and let the rest of the scene stay
coherent around it.

## Language

The description is always in English, whatever language the request arrives in. The
only exception is text shown inside the image, which stays in its own script.

## Output format

- Return the description and nothing else: one continuous paragraph of English, in a single line
starting on the very first character of your reply.
- No JSON, no braces, no key names, no labels, no headings, no markdown, no code fences, and no quotation marks around the paragraph.
- No preamble, no acknowledgement, no explanation, no summary, no notes, no closing remarks, no questions.
- No ratio, no resolution, no pixel count, no timestamps.

Your whole reply is used as the image prompt, so every extra word you add becomes part of the image.
'''

IS_TURBO = MODEL_MODE.startswith("Turbo")
UNET_NAME = "qwen_image_2.1_turbo_int8_convrot.safetensors" if IS_TURBO else "qwen_image_2.1_int8_convrot.safetensors"
CLIP_NAME = "qwen3vl_8b_int8_convrot.safetensors"
VAE_NAME = "qwen_image_2.1_vae_bf16.safetensors"
PE_T2I_NAME = "qwen3.5_9b_qwen_image_2.1_pe_t2i.int8_convrot.safetensors"

def ensure_asset(rel_dir, fname, url):
    target_dir = os.path.join(WORKSPACE, "models", rel_dir)
    os.makedirs(target_dir, exist_ok=True)
    dest = os.path.join(target_dir, fname)
    if not (os.path.exists(dest) and os.path.getsize(dest) > 100000):
        print(f"📥 [@CoinNoin] Downloading missing model: {fname}...")
        subprocess.run([
            "aria2c", "--console-log-level=error", "--summary-interval=10",
            "-c", "-x", "16", "-s", "16", "-k", "1M",
            "-o", fname, url, "-d", target_dir
        ], check=True)

ensure_asset(
    "diffusion_models", UNET_NAME,
    f"https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/{UNET_NAME}"
)
if REFINE_PROMPT:
    ensure_asset(
        "text_encoders", PE_T2I_NAME,
        f"https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/{PE_T2I_NAME}"
    )

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server...")
def ensure_comfyui():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req, timeout=3)
        print("   🟢 ComfyUI server is already running.")
    except Exception:
        print("   🚀 Starting ComfyUI server with --lowvram for T4...")
        subprocess.Popen(
            [sys.executable, "main.py", "--port", "8188", "--lowvram"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        for _ in range(90):
            try:
                urllib.request.urlopen(req, timeout=3)
                print("   🟢 ComfyUI server is ready!")
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("ComfyUI server failed to start within timeout.")

ensure_comfyui()
clear_output()

print(f"⚙️ [@CoinNoin] T2I | Mode: {MODEL_MODE} | PE: {REFINE_PROMPT} | Ratio: {ASPECT_RATIO} ({MEGAPIXELS}MP) | Seed: {SEED}")

# Build API prompt matching image_qwen_image_2_1_turbo_t2i.json & image_qwen_image_2_1_t2i.json
api_prompt = {
    "13": {
        "class_type": "ResolutionSelector",
        "inputs": {
            "aspect_ratio": ASPECT_RATIO,
            "megapixels": MEGAPIXELS,
            "multiple": MULTIPLE
        }
    },
    "451": {
        "class_type": "UNETLoader",
        "inputs": {
            "unet_name": UNET_NAME,
            "weight_dtype": "default"
        }
    },
    "453": {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": CLIP_NAME,
            "type": "qwen_image",
            "device": "default"
        }
    },
    "454": {
        "class_type": "VAELoader",
        "inputs": {
            "vae_name": VAE_NAME
        }
    },
    "456": {
        "class_type": "EmptyLatentImage",
        "inputs": {
            "width": ["13", 0],
            "height": ["13", 1],
            "batch_size": 1
        }
    },
    "490": {
        "class_type": "QwenImage21Cache",
        "inputs": {
            "model": ["451", 0],
            "device": CACHE_DEVICE,
            "dtype": CACHE_DTYPE
        }
    }
}

# Prompt Enhancer Branch (exact official T2I PE graph when enabled)
if REFINE_PROMPT:
    api_prompt["473"] = {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": PE_T2I_NAME,
            "type": "stable_diffusion",
            "device": "default"
        }
    }
    api_prompt["475"] = {
        "class_type": "PrimitiveStringMultiline",
        "inputs": {"value": T2I_SYSTEM_PROMPT}
    }
    api_prompt["471"] = {
        "class_type": "TextGenerate",
        "inputs": {
            "clip": ["473", 0],
            "system_prompt": ["475", 0],
            "prompt": PROMPT,
            "max_length": PE_MAX_LENGTH,
            "sampling_mode": "on",
            "sampling_mode.temperature": 1.0,
            "sampling_mode.top_k": 20,
            "sampling_mode.top_p": 0.95,
            "sampling_mode.min_p": 0.0,
            "sampling_mode.repetition_penalty": 1.0,
            "sampling_mode.presence_penalty": 1.5,
            "sampling_mode.seed": SEED,
            "thinking": THINKING_MODE,
            "use_default_template": True,
            "mtp": "auto"
        }
    }
    api_prompt["474"] = {
        "class_type": "ComfySwitchNode",
        "inputs": {
            "on_false": PROMPT,
            "on_true": ["471", 0],
            "switch": True
        }
    }
    api_prompt["472"] = {
        "class_type": "PreviewAny",
        "inputs": {"source": ["474", 0]}
    }
    active_prompt_input = ["472", 0]
else:
    active_prompt_input = PROMPT

api_prompt["452"] = {
    "class_type": "TextEncodeQwenImage21",
    "inputs": {
        "clip": ["453", 0],
        "prompt": active_prompt_input,
        "negative_prompt": NEGATIVE_PROMPT,
        "resolution": 1024
    }
}

if IS_TURBO:
    # Exact Turbo 8-step ManualSigmas + SamplerCustomAdvanced pipeline
    api_prompt["484"] = {
        "class_type": "ManualSigmas",
        "inputs": {
            "sigmas": "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0"
        }
    }
    api_prompt["487"] = {
        "class_type": "KSamplerSelect",
        "inputs": {"sampler_name": SAMPLER_NAME}
    }
    api_prompt["488"] = {
        "class_type": "RandomNoise",
        "inputs": {"noise_seed": SEED}
    }
    api_prompt["489"] = {
        "class_type": "CFGGuider",
        "inputs": {
            "model": ["490", 0],
            "positive": ["452", 0],
            "negative": ["452", 1],
            "cfg": CFG
        }
    }
    api_prompt["486"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["488", 0],
            "guider": ["489", 0],
            "sampler": ["487", 0],
            "sigmas": ["484", 0],
            "latent_image": ["456", 0]
        }
    }
    sampled_latent = ["486", 0]
else:
    # Exact Base KSampler pipeline
    api_prompt["458"] = {
        "class_type": "KSampler",
        "inputs": {
            "model": ["490", 0],
            "positive": ["452", 0],
            "negative": ["452", 1],
            "latent_image": ["456", 0],
            "seed": SEED,
            "steps": STEPS,
            "cfg": CFG,
            "sampler_name": SAMPLER_NAME,
            "scheduler": SCHEDULER,
            "denoise": 1.0
        }
    }
    sampled_latent = ["458", 0]

api_prompt["457"] = {
    "class_type": "VAEDecode",
    "inputs": {
        "samples": sampled_latent,
        "vae": ["454", 0]
    }
}

api_prompt["461"] = {
    "class_type": "SaveImageAdvanced",
    "inputs": {
        "images": ["457", 0],
        "filename_prefix": "Qwen_image_2.1",
        "format": "png",
        "format.bit_depth": BIT_DEPTH,
        "format.input_color_space": INPUT_COLOR_SPACE
    }
}

payload = json.dumps({"prompt": api_prompt}).encode("utf-8")
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=payload,
    headers={"Content-Type": "application/json"}
)

print("\n📥 [@CoinNoin] Submitting T2I workflow...")
try:
    with urllib.request.urlopen(req) as resp:
        prompt_id = json.loads(resp.read().decode("utf-8"))["prompt_id"]
except urllib.error.HTTPError as e:
    print(f"❌ ComfyUI API Error: {e.read().decode('utf-8')}")
    raise

print("✨ Generating image...")
outputs = {}
while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as h_resp:
            hist = json.loads(h_resp.read().decode("utf-8"))
        if prompt_id in hist:
            outputs = hist[prompt_id].get("outputs", {})
            break
    except Exception:
        pass
    time.sleep(2)

if REFINE_PROMPT and "472" in outputs and "text" in outputs["472"]:
    print(f"\n📝 Refined Prompt:\n{outputs['472']['text'][0]}\n")

saved_images = []
for node_id, node_out in outputs.items():
    if "images" in node_out:
        for item in node_out["images"]:
            fname = item.get("filename")
            subf = item.get("subfolder", "")
            if fname:
                img_path = os.path.join(OUTPUT_DIR, subf, fname) if subf else os.path.join(OUTPUT_DIR, fname)
                if os.path.exists(img_path):
                    saved_images.append(img_path)
                    print(f"\033[92m✓ [@CoinNoin] Saved Output: {img_path}\033[0m")
                    display(IPImage(filename=img_path))

if AUTO_DOWNLOAD and saved_images:
    print("\n📥 [@CoinNoin] Downloading output image(s)...")
    for ip in saved_images:
        files.download(ip)

In [ ]:
#@title 4. Image Edit
#@markdown ### Easy Guide:
#@markdown 1. Keep **UPLOAD_NEW_IMAGES** checked and run this cell to upload **1 image** (or select **multiple images up to 10** at once for multi-image edits).
#@markdown 2. To re-run the same uploaded image(s) with a new prompt, uncheck **UPLOAD_NEW_IMAGES** — filenames are auto-detected dynamically.
#@markdown 3. For multi-image edits, refer to them in **PROMPT** as `<image1>`, `<image2>`, etc. (`<image1>` is the main edit target).
#@markdown 4. Pick **Turbo (Fast 8-Step)** for speed on T4 or **Base (25-Step)** for standard sampling. Keep `CFG = 1.0`.
#@markdown 5. **Out of Memory (OOM) on T4?** Set `RESOLUTION` to `1024` (or `768`) and/or switch `CACHE_DEVICE` to `cpu` or `off`.

# --- Input Images ---
UPLOAD_NEW_IMAGES = True #@param {type:"boolean"}

# --- Model Mode ---
MODEL_MODE = "Turbo (Fast 8-Step)" #@param ["Turbo (Fast 8-Step)", "Base (25-Step)"]

# --- Prompt Settings ---
PROMPT = "Transform the atmosphere to a bright, sun-drenched midday, with overhead sunlight, crisp, harsh shadows, and vibrant natural colors." #@param {type:"string"}
NEGATIVE_PROMPT = "" #@param {type:"string"}

# --- Prompt Enhancer (Optional) ---
REFINE_PROMPT = False #@param {type:"boolean"}
THINKING_MODE = False #@param {type:"boolean"}
PE_MAX_LENGTH = 4096 #@param {type:"integer"}

# --- Size Settings ---
RESOLUTION = 0 #@param {type:"slider", min:0, max:2048, step:32}
CUSTOM_SIZE = False #@param {type:"boolean"}
WIDTH = 1024 #@param {type:"slider", min:256, max:2048, step:32}
HEIGHT = 1024 #@param {type:"slider", min:256, max:2048, step:32}

# --- Sampler & Cache Settings ---
STEPS = 8 #@param {type:"slider", min:1, max:50, step:1}
#@markdown *Note: `STEPS` and `SCHEDULER` apply to **Base** mode. **Turbo** mode uses its official 8-step `ManualSigmas` schedule.*
CFG = 1.0 #@param {type:"number"}
SAMPLER_NAME = "euler" #@param ["euler", "euler_ancestral", "dpmpp_2m", "dpmpp_2m_sde", "uni_pc", "res_multistep"] {allow-input: true}
SCHEDULER = "simple" #@param ["simple", "normal", "karras", "beta", "sgm_uniform"] {allow-input: true}
SEED = 0 #@param {type:"integer"}
CACHE_DEVICE = "auto" #@param ["auto", "gpu", "cpu", "off"]
CACHE_DTYPE = "default" #@param ["default", "int8", "int4"]

# --- Output Settings ---
BIT_DEPTH = "8-bit" #@param ["8-bit", "16-bit"]
INPUT_COLOR_SPACE = "sRGB" #@param ["sRGB"]
AUTO_DOWNLOAD = True #@param {type:"boolean"}

import sys
import os
import json
import time
import random
import subprocess
import urllib.request
from google.colab import files
from PIL import Image
from IPython.display import display, Image as IPImage, clear_output

WORKSPACE = "/content/ComfyUI"
INPUT_DIR = os.path.join(WORKSPACE, "input")
OUTPUT_DIR = os.path.join(WORKSPACE, "output")
os.makedirs(INPUT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.chdir(WORKSPACE)

I2I_SYSTEM_PROMPT = '''# Edit Prompt Enhancer — General (v2, 精简版)

**FIRST — there are TWO separate language decisions. Do NOT conflate them.**

**(A) Language of the rewritten prompt's DESCRIPTIVE prose — every word OUTSIDE double quotes (the description you write for the diffusion model, NOT the text painted into the image). This decision is final and non-negotiable:**
- User instruction is in Chinese → write the description in Chinese.
- User instruction is in English → write the description in English.
- User instruction is in ANY other language (Japanese, Korean, French, Spanish, Thai, etc.) → write the description in English.

**(B) Language of the TEXT THAT WILL BE RENDERED INTO THE OUTPUT IMAGE — the content INSIDE double quotes. Decide it in this strict priority order:**
1. If the user's instruction gives the exact text to write, OR names a target language for the text (e.g. "改成'夏日特惠'", "把标题写成英文", "add a Japanese title", "write the caption in Thai") → render exactly that text / in exactly that specified language.
2. Otherwise, if the input image already contains text → render in the DOMINANT language of the image's existing text — even when the instruction is written in a different language.
3. Otherwise (the image contains no text AND the instruction names no target language) → render in the language of the user's instruction itself — including Japanese, Korean, Thai, Arabic, French, etc. Do NOT force it to English.
Worked example: image is mostly Thai, instruction is in English asking to add/redesign a title without giving the exact words or a language → the rendered (quoted) text must be **Thai** (the image's dominant language), while the surrounding description (A) is still written in English.

Two reinforcements on decision (B): all rendered (quoted) text must be **monolingual** — do not mix Chinese and English inside the quotes and do not emit a bilingual pair unless the user explicitly asks for one. And **genre never overrides input language**: a "spec sheet / cinematic data-document / storyboard / technical parameter" look is achieved through layout and typography, NOT by switching rendered labels to English — every header, label, and caption stays in the decided language (standardized units and user-given proper nouns may remain Latin).

You are an expert at clarifying image editing instructions. Given a user's vague or ambiguous edit instruction and the input image(s), rewrite it into a precise, unambiguous, actionable editing directive. An input image is ALWAYS present — this is always an image-editing task, never text-to-image from nothing.

## Core Objective

Rewrite the instruction so a downstream image-editing model can execute it without guessing — anchored on what the input image(s) actually show, faithful to the user's intent, inventing nothing.

**How much you build is intent-branched.** When the user wants *this picture changed* (a local object/attribute/background edit, a text or UI edit, a quality or style change, a viewpoint/canvas transform), clarify and constrain: say exactly what changes, and let everything else stand. When the user wants *a new picture of this subject* (placing a subject in a new scene, compositing across images, a photo-shoot or poster or infographic built from a reference), construct actively: design the scene, lighting, composition and layout to a professional standard. Scale the elaboration to what was asked — a plain placement stays restrained, a styled shoot or a publication-grade poster is built out fully.

## The Governing Principle — Attribute Disentanglement at Full Strength

**Edit exactly the attribute(s) the user named, push each to a strong and unmistakable degree, and hold everything else at input fidelity.**

Both halves matter, and the two failure modes are symmetric:

- **Leakage** — touching what the user did not name (a sharpen that re-grades color, an upscale that reframes, a style change that drifts a face, an outfit swap that drops an accessory, a background change that "helpfully" cleans up something unmentioned).
- **Under-editing** — an output a viewer could mistake for the unedited input, because the requested change was applied faintly.

Preservation locks **content, never edit strength**. Recognizability is bought by naming what stays fixed, not by holding the effect back.

## What to Anchor, What to Decide

**Anchor on the image.** Every spatial, tonal and contextual claim comes from what is visibly there. If you are unsure a detail exists, leave it out — a preserved element described at a higher level of abstraction is always safer than an invented specific.

**Say what stays, without repainting it.** Name the untargeted content by type, position and role rather than describing its appearance, and prefer one blanket preservation clause over walking the frame. A preservation description reads to the model as a generation instruction: the more concretely you describe something you meant to keep, the more likely it drifts. Describe appearance concretely only for what you are actually changing, or when it is the only way to disambiguate between similar objects.

**Identity is the hardest invariant.** A person's facial identity and the personal accessories that make them recognizable; a product's exact design, markings and count; and the input's rendering medium (photograph, anime, illustration, sketch, 3D render, painting) all survive every edit unless the user explicitly targets them. When identity comes from a reference image, point at that image rather than describing features in words — verbal descriptions make the model regenerate and degrade the likeness.

**Resolve ambiguity, then commit.** Turn vague intent, imprecise spatial reference and unparameterized style words into something concrete and observable. Translate abstract quality language into the visual properties it implies. Where the instruction offers alternatives or contradicts itself, pick the most reasonable reading and state it as a decision. Keep the user's own action verb, spatial relations and described state intact, and treat anything they asked to preserve as absolute. Preserve creative or physically impossible intent rather than correcting it.

**Only what was asked.** Do not add operations the user did not request, and do not clean up unmentioned defects, overlays or clutter however prominent they look. When an edit removes, moves or reveals something, say enough about the newly exposed region that the result stays physically coherent.

**Text in the image is literal.** Whenever readable text will appear in the output, commit to the exact characters — every element, quoted, nothing summarized or abbreviated away. Text you cannot commit to should not be added at all. Match the typography and language the input establishes unless the user asks otherwise. When the operation extends the canvas outward, name it as outpainting explicitly.

**Write it as an instruction.** Lead with the operation, not a description of the finished picture, and write from the perspective of someone holding only the input image(s).

## Thinking Process

Before emitting the final prompt, reason through: what the image(s) actually contain (including a complete reading of any text present); what the user is asking for and which attributes that names; what must therefore stay fixed; and finally the composed directive. Close with a check that every visible element is either the target of the edit or covered by what stays fixed, that the requested change is unmistakable, that nothing outside the target was touched, and that every quoted string obeys language decision (B).

## Image Reference Rules

For Multi-Image Input (N >= 2), the rewritten instruction MUST use `<image1>`, `<image2>`, ... to refer to each input image. Do not use natural language references like "图1", "第一张图", "the first image", or "image A". This tagging format is mandatory and non-negotiable. For single-image input (N = 1), do NOT use tags — refer to the image naturally ("图像", "图片中", "the image").

State each image's role explicitly — which one is the canvas whose composition and untargeted content survive, and which supply material to transfer — and say what is taken from each. For scene generation with no canvas (合影/合照 and the like), all images serve as identity sources. Describe every referenced image individually; never compress several into a range or a group to avoid describing them one by one.

## Output Format
Output ONLY the rewritten editing instruction itself: one single continuous paragraph of plain text.

Formatting rules:
- The entire rewritten prompt must be a single continuous paragraph with NO line breaks or newline characters.
- The response must contain nothing except the instruction itself: no object or array wrapper, no field names or key-value pairs, no headings, labels, or prefaces, no greetings, explanations, or commentary, no markdown, and no code fences. Begin with the first word of the instruction and end with its last word.
- Do not wrap the whole instruction in quotes: double quotes are reserved for text that will be RENDERED in the image (next rule).
- All text that should appear as visible, readable content in the output image must be enclosed in double quotes (""). Descriptive or structural language that does not appear as rendered text should NOT be quoted.
- **Never include any resolution or aspect ratio information** (e.g., "2:3", "16:9", "1920x1080", "2K", "4K"). Output size is decided downstream, not by you.
- Write it out in full — no ellipsis, no truncation.
- State requirements affirmatively ("保持背景与输入图完全一致") rather than as prohibitions ("禁止改变背景"). Standard preservation phrasing "保持/保留[X]不变" is fine.
- Be precise and decisive: no hedging, no unresolved alternatives, no vague degree words left unresolved.
- The descriptive prose (outside double quotes) follows language decision (A); the text rendered inside the image (inside double quotes) follows language decision (B). Retain proper nouns and domain-specific terms in their original language, placed in English double quotes.
- **Language-purge self-check (do this last)**: re-scan every double-quoted string — the text that will be RENDERED in the image — and enforce language decision (B). No quoted string may mix Chinese and English, form a bilingual pair, or carry a parenthetical translation gloss unless the user explicitly asked. Standardized units and user-given proper nouns may remain Latin.

The user's edit instruction to rewrite is:'''

IMAGE_EXTS = (".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff")

def get_latest_input_image():
    imgs = [
        os.path.join(INPUT_DIR, f)
        for f in os.listdir(INPUT_DIR)
        if f.lower().endswith(IMAGE_EXTS)
    ]
    if not imgs:
        return []
    return [os.path.basename(max(imgs, key=os.path.getmtime))]

def upload_images_to_input():
    print("📤 [@CoinNoin] Select 1 or more images (up to 10) to upload...")
    uploaded = files.upload()
    if not uploaded:
        return []
    saved = []
    for up_name, raw_bytes in list(uploaded.items())[:10]:
        clean_name = os.path.basename(up_name)
        dst_path = os.path.join(INPUT_DIR, clean_name)
        with open(dst_path, "wb") as f:
            f.write(raw_bytes)
        cwd_copy = os.path.join(os.getcwd(), up_name)
        if os.path.exists(cwd_copy) and os.path.abspath(cwd_copy) != os.path.abspath(dst_path):
            os.remove(cwd_copy)
        saved.append(clean_name)
        print(f"   ✅ Uploaded <image{len(saved)}>: {clean_name}")
    return saved

# Dynamically resolve uploaded image filenames
IMAGE_FILENAMES = []
if UPLOAD_NEW_IMAGES:
    IMAGE_FILENAMES = upload_images_to_input()

if not IMAGE_FILENAMES:
    prev = globals().get("LAST_EDIT_IMAGES", [])
    IMAGE_FILENAMES = [f for f in prev if os.path.exists(os.path.join(INPUT_DIR, f))]

if not IMAGE_FILENAMES:
    IMAGE_FILENAMES = get_latest_input_image()

if not IMAGE_FILENAMES:
    print("⚠️ No image found in input folder. Please upload at least 1 image now.")
    IMAGE_FILENAMES = upload_images_to_input()

if not IMAGE_FILENAMES:
    raise RuntimeError("❌ No image uploaded. Please re-run this cell and select an image file.")

LAST_EDIT_IMAGES = IMAGE_FILENAMES

IS_TURBO = MODEL_MODE.startswith("Turbo")
UNET_NAME = "qwen_image_2.1_turbo_int8_convrot.safetensors" if IS_TURBO else "qwen_image_2.1_int8_convrot.safetensors"
CLIP_NAME = "qwen3vl_8b_int8_convrot.safetensors"
VAE_NAME = "qwen_image_2.1_vae_bf16.safetensors"
PE_I2I_NAME = "qwen3.5_9b_qwen_image_2.1_pe_i2i.int8_convrot.safetensors"

def ensure_asset(rel_dir, fname, url):
    target_dir = os.path.join(WORKSPACE, "models", rel_dir)
    os.makedirs(target_dir, exist_ok=True)
    dest = os.path.join(target_dir, fname)
    if not (os.path.exists(dest) and os.path.getsize(dest) > 100000):
        print(f"📥 [@CoinNoin] Downloading missing model: {fname}...")
        subprocess.run([
            "aria2c", "--console-log-level=error", "--summary-interval=10",
            "-c", "-x", "16", "-s", "16", "-k", "1M",
            "-o", fname, url, "-d", target_dir
        ], check=True)

ensure_asset(
    "diffusion_models", UNET_NAME,
    f"https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/{UNET_NAME}"
)
if REFINE_PROMPT:
    ensure_asset(
        "text_encoders", PE_I2I_NAME,
        f"https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/text_encoders/{PE_I2I_NAME}"
    )

if SEED == 0:
    SEED = random.randint(1, 1125899906842624)

print("🔌 [@CoinNoin] Checking ComfyUI Server...")
def ensure_comfyui():
    req = urllib.request.Request("http://127.0.0.1:8188/system_stats")
    try:
        urllib.request.urlopen(req, timeout=3)
        print("   🟢 ComfyUI server is already running.")
    except Exception:
        print("   🚀 Starting ComfyUI server with --lowvram for T4...")
        subprocess.Popen(
            [sys.executable, "main.py", "--port", "8188", "--lowvram"],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        for _ in range(90):
            try:
                urllib.request.urlopen(req, timeout=3)
                print("   🟢 ComfyUI server is ready!")
                break
            except Exception:
                time.sleep(2)
        else:
            raise RuntimeError("ComfyUI server failed to start within timeout.")

ensure_comfyui()
clear_output()

print(f"⚙️ [@CoinNoin] Image Edit | Mode: {MODEL_MODE} | PE: {REFINE_PROMPT} | Seed: {SEED}")
for idx, fname in enumerate(IMAGE_FILENAMES, 1):
    print(f"   🖼️ <image{idx}>: {fname}")
    p = os.path.join(INPUT_DIR, fname)
    if os.path.exists(p):
        im = Image.open(p)
        display(im.resize((220, int(220 * im.height / im.width))))

# Build API prompt matching image_qwen_image_2_1_turbo_image_edit.json & image_qwen_image_2_1_image_edit.json
api_prompt = {
    "451": {
        "class_type": "UNETLoader",
        "inputs": {
            "unet_name": UNET_NAME,
            "weight_dtype": "default"
        }
    },
    "453": {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": CLIP_NAME,
            "type": "qwen_image",
            "device": "default"
        }
    },
    "454": {
        "class_type": "VAELoader",
        "inputs": {
            "vae_name": VAE_NAME
        }
    },
    "456": {
        "class_type": "EmptyLatentImage",
        "inputs": {
            "width": WIDTH,
            "height": HEIGHT,
            "batch_size": 1
        }
    },
    "469": {
        "class_type": "QwenImage21Cache",
        "inputs": {
            "model": ["451", 0],
            "device": CACHE_DEVICE,
            "dtype": CACHE_DTYPE
        }
    }
}

# Dynamic LoadImage nodes for every uploaded image (image_1 .. image_10)
load_node_ids = []
for idx, fname in enumerate(IMAGE_FILENAMES, 1):
    nid = f"load_{idx}"
    api_prompt[nid] = {
        "class_type": "LoadImage",
        "inputs": {"image": fname}
    }
    load_node_ids.append(nid)

# Prompt Enhancer Branch (exact official I2I PE graph when enabled)
if REFINE_PROMPT:
    batch_inputs = {f"images.image{i}": [nid, 0] for i, nid in enumerate(load_node_ids)}
    api_prompt["485"] = {
        "class_type": "BatchImagesNode",
        "inputs": batch_inputs
    }
    api_prompt["477"] = {
        "class_type": "CLIPLoader",
        "inputs": {
            "clip_name": PE_I2I_NAME,
            "type": "qwen_image",
            "device": "default"
        }
    }
    api_prompt["479"] = {
        "class_type": "PrimitiveStringMultiline",
        "inputs": {"value": I2I_SYSTEM_PROMPT}
    }
    api_prompt["500"] = {
        "class_type": "TextGenerate",
        "inputs": {
            "clip": ["477", 0],
            "image": ["485", 0],
            "system_prompt": ["479", 0],
            "prompt": PROMPT,
            "max_length": PE_MAX_LENGTH,
            "sampling_mode": "on",
            "sampling_mode.temperature": 1.0,
            "sampling_mode.top_k": 20,
            "sampling_mode.top_p": 0.95,
            "sampling_mode.min_p": 0.0,
            "sampling_mode.repetition_penalty": 1.0,
            "sampling_mode.presence_penalty": 0.0,
            "sampling_mode.seed": SEED,
            "thinking": THINKING_MODE,
            "use_default_template": True,
            "mtp": "auto"
        }
    }
    api_prompt["484"] = {
        "class_type": "ComfySwitchNode",
        "inputs": {
            "on_false": PROMPT,
            "on_true": ["500", 0],
            "switch": True
        }
    }
    api_prompt["480"] = {
        "class_type": "PreviewAny",
        "inputs": {"source": ["484", 0]}
    }
    active_prompt_input = ["480", 0]
else:
    active_prompt_input = PROMPT

encode_inputs = {
    "clip": ["453", 0],
    "vae": ["454", 0],
    "prompt": active_prompt_input,
    "negative_prompt": NEGATIVE_PROMPT,
    "resolution": RESOLUTION
}
for idx, nid in enumerate(load_node_ids, 1):
    encode_inputs[f"images.image_{idx}"] = [nid, 0]

api_prompt["474"] = {
    "class_type": "TextEncodeQwenImage21",
    "inputs": encode_inputs
}

api_prompt["468"] = {
    "class_type": "ComfySwitchNode",
    "inputs": {
        "on_false": ["474", 2],
        "on_true": ["456", 0],
        "switch": CUSTOM_SIZE
    }
}

if IS_TURBO:
    # Exact Turbo 8-step ManualSigmas + SamplerCustomAdvanced pipeline
    api_prompt["502"] = {
        "class_type": "ManualSigmas",
        "inputs": {
            "sigmas": "1.0, 0.978453, 0.95418, 0.926626, 0.89508, 0.845148, 0.704534, 0.414568, 0"
        }
    }
    api_prompt["503"] = {
        "class_type": "KSamplerSelect",
        "inputs": {"sampler_name": SAMPLER_NAME}
    }
    api_prompt["504"] = {
        "class_type": "RandomNoise",
        "inputs": {"noise_seed": SEED}
    }
    api_prompt["507"] = {
        "class_type": "CFGGuider",
        "inputs": {
            "model": ["469", 0],
            "positive": ["474", 0],
            "negative": ["474", 1],
            "cfg": CFG
        }
    }
    api_prompt["505"] = {
        "class_type": "SamplerCustomAdvanced",
        "inputs": {
            "noise": ["504", 0],
            "guider": ["507", 0],
            "sampler": ["503", 0],
            "sigmas": ["502", 0],
            "latent_image": ["468", 0]
        }
    }
    sampled_latent = ["505", 0]
else:
    # Exact Base KSampler pipeline
    api_prompt["458"] = {
        "class_type": "KSampler",
        "inputs": {
            "model": ["469", 0],
            "positive": ["474", 0],
            "negative": ["474", 1],
            "latent_image": ["468", 0],
            "seed": SEED,
            "steps": STEPS,
            "cfg": CFG,
            "sampler_name": SAMPLER_NAME,
            "scheduler": SCHEDULER,
            "denoise": 1.0
        }
    }
    sampled_latent = ["458", 0]

api_prompt["457"] = {
    "class_type": "VAEDecode",
    "inputs": {
        "samples": sampled_latent,
        "vae": ["454", 0]
    }
}

api_prompt["461"] = {
    "class_type": "SaveImageAdvanced",
    "inputs": {
        "images": ["457", 0],
        "filename_prefix": "Qwen_image_2.1",
        "format": "png",
        "format.bit_depth": BIT_DEPTH,
        "format.input_color_space": INPUT_COLOR_SPACE
    }
}

payload = json.dumps({"prompt": api_prompt}).encode("utf-8")
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=payload,
    headers={"Content-Type": "application/json"}
)

print("\n📥 [@CoinNoin] Submitting Image Edit workflow...")
try:
    with urllib.request.urlopen(req) as resp:
        prompt_id = json.loads(resp.read().decode("utf-8"))["prompt_id"]
except urllib.error.HTTPError as e:
    print(f"❌ ComfyUI API Error: {e.read().decode('utf-8')}")
    raise

print("✨ Editing image...")
outputs = {}
while True:
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:8188/history/{prompt_id}") as h_resp:
            hist = json.loads(h_resp.read().decode("utf-8"))
        if prompt_id in hist:
            outputs = hist[prompt_id].get("outputs", {})
            break
    except Exception:
        pass
    time.sleep(2)

if REFINE_PROMPT and "480" in outputs and "text" in outputs["480"]:
    print(f"\n📝 Refined Prompt:\n{outputs['480']['text'][0]}\n")

saved_images = []
for node_id, node_out in outputs.items():
    if "images" in node_out:
        for item in node_out["images"]:
            fname = item.get("filename")
            subf = item.get("subfolder", "")
            if fname:
                img_path = os.path.join(OUTPUT_DIR, subf, fname) if subf else os.path.join(OUTPUT_DIR, fname)
                if os.path.exists(img_path):
                    saved_images.append(img_path)
                    print(f"\033[92m✓ [@CoinNoin] Saved Output: {img_path}\033[0m")
                    display(IPImage(filename=img_path))

if AUTO_DOWNLOAD and saved_images:
    print("\n📥 [@CoinNoin] Downloading output image(s)...")
    for ip in saved_images:
        files.download(ip)